<a href="https://colab.research.google.com/github/Adrien-Rouillon/Machine-learning---SpotifyDB/blob/main/Notebooks/Delivrable2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOD10 – Project, Deliverable 2 : exploration and pre-processing data
**Membres du groupe :** Adrien ROUILLON, Paul MORAND

**Data set :** Spotify

---

In [27]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
url = "https://raw.githubusercontent.com/Adrien-Rouillon/Machine-learning---SpotifyDB/refs/heads/main/spotify_artist_streaming_2020_2025.csv?token=GHSAT0AAAAAAEJEI4BTVCIBYZR7MBU6DWBW2WZIFRQ"
df = pd.read_csv(url)


Before we need to do any data analysis, we need to mess it up since we chose a perfect dataset.

We gave the prompt from class 2 notebook into Claude. It generated us a "add_noise" function to make the data less clean.

In [28]:
def add_noise(df, seed=42):  # return a messy copy of a clean DataFrame
    rng = np.random.default_rng(seed)  # fixed seed -> same mess every run
    out = df.copy(deep=True)  # work on a deep copy so df is never modified
    num_cols = out.select_dtypes("number").columns.tolist()  # numeric columns available to corrupt
    if len(num_cols) < 2:  # need at least two numeric columns
        raise ValueError("df needs at least 2 numeric columns")  # fail loudly instead of guessing
    miss_cols = num_cols[:2]  # first two numeric columns get missing values
    out_col = num_cols[0]  # first numeric column gets outliers
    text_col = num_cols[2] if len(num_cols) > 2 else num_cols[1]  # third (or second) becomes text
    n = len(out)  # number of original rows

    # --- extreme outliers ---
    out_idx = rng.choice(out.index, size=3, replace=False)  # pick 3 rows to corrupt
    col = out[out_col].astype(float)  # float copy so large values fit
    col.loc[out_idx] = col.mean() + 20 * col.std() * rng.choice([-1, 1], size=3)  # push Â±20 std from the mean
    out[out_col] = col  # write outliers back

    # --- ~5% missing values in two numeric columns ---
    for c in miss_cols:  # handle each target column
        nan_idx = rng.choice(out.index, size=max(1, round(0.05 * n)), replace=False)  # 5% random rows
        out[c] = out[c].astype(float)  # ints can't hold NaN, so cast to float
        out.loc[nan_idx, c] = np.nan  # blank them out

    # --- inconsistent categorical column ---
    base = rng.choice(["Red", "Green", "Blue"], size=n)  # clean underlying categories
    variants = [str.lower, str.upper, str.title, lambda s: f" {s}", lambda s: f"{s}  "]  # case/spacing styles
    out["color"] = [variants[rng.integers(len(variants))](v) for v in base]  # apply a random style per row

    # --- numeric column stored as text with a stray 'unknown' ---
    out[text_col] = out[text_col].map(lambda v: "" if pd.isna(v) else str(v)).astype(object)  # numbers -> strings
    out.loc[rng.choice(out.index), text_col] = "unknown"  # one non-numeric token

    # --- 5 exact duplicate rows ---
    dup_rows = out.loc[rng.choice(out.index, size=5, replace=False)]  # copy 5 existing (already messy) rows
    out = pd.concat([out, dup_rows], ignore_index=True)  # append them -> exact duplicates
    out = out.sample(frac=1, random_state=seed).reset_index(drop=True)  # shuffle so dupes aren't at the end

    return out  # the messy copy

df = add_noise(df)

## Étape 1 – Exploration (EDA)



---



In [29]:
df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,loudness_category,key_name,mode_name,is_explicit_bool,release_quarter,is_weekend_release,log_stream_count,upbeat_score,artist_track_count,color
0,1c22dd2a-4646-47a1-b354-2fc81b05088a,Electric Fire,Ethereal,The Broken Stories,2020-05-31,Pop,178551.0,33.0,0.6422,0.6088,...,Moderate,C#,Major,True,Q2,True,9.7061,0.6007,3232,GREEN
1,991b08a9-b94a-4a87-a397-6b547bfa2626,Golden Mirrors,IsleEyre,Endless Keys,2022-10-14,Pop,NaN,17.0,0.6503,0.5237,...,Moderate,G#,Major,False,Q4,False,7.9659,0.5485,5271,Blue
2,2939f47b-f4e1-4ce1-bcce-47382fae8b38,Walking Echoes,IsleEyre,Tapes,2022-05-10,Pop,226433.0,24.0,0.5826,0.8425,...,Moderate,D#,Minor,False,Q2,False,10.2064,0.6179,5271,Red
3,0fb1ce22-3de6-432f-97fd-cb20d2dc175a,Paper Gentle Shadows,IsleEyre,Burning Wings,2020-11-29,Soul,204174.0,26.0,0.5797,0.4981,...,Moderate,D#,Minor,False,Q4,True,9.7636,0.5348,5271,BLUE
4,3b11fe9f-0067-4c90-8723-79be21855941,Heavy Shadows,Knot Hart,Fading Lights,2021-08-17,R&B,202885.0,17.0,0.7227,0.3697,...,Moderate,E,Minor,True,Q3,False,9.1829,0.5097,12111,blue


In [30]:
df.shape

(50005, 34)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 82 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50000 non-null  object 
 1   track_name           50000 non-null  object 
 2   artist_name          50000 non-null  object 
 3   album_name           50000 non-null  object 
 4   release_date         50000 non-null  object 
 5   duration_ms          50000 non-null  int64  
 6   popularity           50000 non-null  int64  
 7   danceability         50000 non-null  float64
 8   energy               50000 non-null  float64
 9   key                  50000 non-null  int64  
 10  loudness             50000 non-null  float64
 11  mode                 50000 non-null  int64  
 12  instrumentalness     50000 non-null  float64
 13  tempo                50000 non-null  float64
 14  stream_count         50000 non-null  int64  
 15  explicit             0 non-null     

In [20]:
df.describe()

,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,tempo,stream_count,explicit,release_year,release_month,duration_minutes,mode_name,log_stream_count,upbeat_score,artist_track_count
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000,50000.000000,50000.000000,50000.000000,50000.000000,5.000000e+04,0.0,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,209814.455560,28.136820,0.661236,0.631853,5.47086,-7.669547,0.650280,0.055166,119.067461,1.183789e+05,NaN,2022.499620,6.528880,3.496915,0.650280,9.835507,0.601615,4052.066320
std,44646.016547,15.999501,0.164143,0.168741,3.45610,2.621917,0.476886,0.095083,19.647807,5.658443e+05,NaN,1.709953,3.460926,0.744101,0.476886,1.793877,0.105076,4817.780532
min,90000.000000,0.000000,0.000000,0.000000,0.00000,-18.620000,0.000000,0.000000,60.000000,1.000000e+02,NaN,2020.000000,1.000000,1.500000,0.000000,4.615100,0.057800,2.000000
25%,179753.000000,16.000000,0.562400,0.523400,2.00000,-9.440000,0.000000,0.010300,105.990000,4.979750e+03,NaN,2021.000000,4.000000,3.000000,0.000000,8.513350,0.542100,181.000000
50%,209689.000000,26.000000,0.675000,0.633400,5.00000,-7.660000,1.000000,0.025700,118.575000,1.595500e+04,NaN,2023.000000,7.000000,3.490000,1.000000,9.677600,0.606400,1486.000000
75%,239708.250000,39.000000,0.775400,0.744700,8.00000,-5.880000,1.000000,0.057600,131.350000,6.068550e+04,NaN,2024.000000,10.000000,4.000000,1.000000,11.013525,0.671100,5271.000000
max,388117.000000,92.000000,1.000000,1.000000,11.00000,0.000000,1.000000,1.000000,220.000000,4.334502e+07,NaN,2025.000000,12.000000,6.470000,1.000000,17.584700,0.915400,12111.000000




---



**Observation:**

.head() : shows us the first five rows of the data base.

.shape : give us the size of the data base (rows, columns)

## Étape 2 – Missing values



---



In [31]:
df.isna().sum()

,0
track_id,0
track_name,0
artist_name,0
album_name,0
release_date,0
genre,0
duration_ms,2500
popularity,2500
danceability,0
energy,0


**Observation:**

.isna() : Returns us the table with either True or False depending if the cell is empty or not.

.sum() : Counts each True in a column.

## Étape 3 – Encoding



---



<style>
  table {
    table-layout: fixed;
    width: 100%;
  }
  table th, table td {
    word-wrap: break-word;
    overflow-wrap: break-word;
    padding: 10px 12px !important;
  }
</style>

<table>
  <thead>
    <tr>
      <th style="width: 30%;">Columns</th>
      <th style="width: 25%;">Decision</th>
      <th style="width: 45%;">Why</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>track_name<br>album_name<br>artist_name<br>release_date</td>
      <td>Removed</td>
      <td>Almost a unique value per row, so nothing to learn. release_year and release_month already carry the date info.</td>
    </tr>
    <tr>
      <td>popularity_category</td>
      <td>Removed</td>
      <td>It is derived from popularity, your target. Keeping it would be cheating (data leakage).</td>
    </tr>
    <tr>
      <td>duration_minutes<br>key_name<br>mode_name<br>is_explicit_bool<br>loudness_category<br>release_quarter<br>log_stream_count</td>
      <td>Removed</td>
      <td>Duplicates of columns already present. I keep stream_count; you can take log_stream_count instead, but not both.</td>
    </tr>
    <tr>
      <td>explicit<br>is_weekend_release</td>
      <td>bool &rarr; 0/1</td>
      <td>Already binary.</td>
    </tr>
    <tr>
      <td>mode</td>
      <td>Nothing to do</td>
      <td>Already in 0/1.</td>
    </tr>
    <tr>
      <td>key<br>genre<br>country<br>release_day_of_week<br>label</td>
      <td>One-hot (get_dummies)</td>
      <td>Unordered categories.</td>
    </tr>
  </tbody>
</table>

In [32]:
for c in df.select_dtypes(include="object").columns:
    print(f"{c:22s} {df[c].nunique():>6} unique values")
#Generated code by AI to see unique values we have to treat.

track_id                50000 unique values
track_name               8895 unique values
artist_name               500 unique values
album_name               4019 unique values
release_date             2192 unique values
genre                      20 unique values
danceability             7626 unique values
country                    30 unique values
label                      30 unique values
release_day_of_week         7 unique values
popularity_category         4 unique values
loudness_category           3 unique values
key_name                   12 unique values
mode_name                   2 unique values
release_quarter             4 unique values
color                      15 unique values


In [33]:
#This is all the columns we have to drop.

to_drop = [
    "track_name", "album_name", "artist_name", "release_date",              #text with unique value
    "popularity_category",                                                  #Leak : popularity drift
    "duration_minutes", "key_name", "mode_name", "is_explicit_bool",
    "loudness_category", "release_quarter", "log_stream_count",             #doubles
]

df = df.drop(columns=[c for c in to_drop if c in df.columns])
df.head()

,track_id,genre,duration_ms,popularity,danceability,energy,key,loudness,mode,instrumentalness,...,country,explicit,label,release_year,release_month,release_day_of_week,is_weekend_release,upbeat_score,artist_track_count,color
0,1c22dd2a-4646-47a1-b354-2fc81b05088a,Pop,178551.0,33.0,0.6422,0.6088,1,-9.76,1,0.0020,...,KR,True,Cipher Music,2020,5,Sunday,True,0.6007,3232,GREEN
1,991b08a9-b94a-4a87-a397-6b547bfa2626,Pop,NaN,17.0,0.6503,0.5237,8,-6.02,1,0.0653,...,BR,False,Cipher Music,2022,10,Friday,False,0.5485,5271,Blue
2,2939f47b-f4e1-4ce1-bcce-47382fae8b38,Pop,226433.0,24.0,0.5826,0.8425,3,-6.02,0,0.0832,...,DE,False,Independent,2022,5,Tuesday,False,0.6179,5271,Red
3,0fb1ce22-3de6-432f-97fd-cb20d2dc175a,Soul,204174.0,26.0,0.5797,0.4981,3,-11.41,0,0.0194,...,KR,False,Columbia Records,2020,11,Sunday,True,0.5348,5271,BLUE
4,3b11fe9f-0067-4c90-8723-79be21855941,R&B,202885.0,17.0,0.7227,0.3697,4,-10.87,0,0.0023,...,US,True,Self-Released,2021,8,Tuesday,False,0.5097,12111,blue


In [ ]:
for c in ["explicit", "is_weekend_release"]:
    df[c] = df[c].astype(int)

## Étape 4 – Scaling



---

